# CascadeGuard GRPO Local Training Notebook

This notebook is configured to train against the local `cascade_guard` package in this checkout, not the deployed Hugging Face Space. Use it to validate the new GRPO reward mode, legal-action API, `isolate` action, stress-imminence prompt, and grader updates before pushing anything.

What it does:

1. Installs dependencies and `pip install -e` from your local repo.
2. Verifies the local environment and optional local OpenEnv server.
3. Builds GRPO training states from local `CascadeEnvironment` with `reward_mode="grpo"` and `training_mode=True`.
4. Trains a LoRA policy with TRL GRPO across the configured task set.
5. Evaluates heuristic, optional LLM, and GRPO policies locally.

You do not need to start the server for training because training uses the environment in-process. A server launcher cell is included only for API/OpenEnv smoke testing.


## 1. Runtime Settings

Set Colab runtime to GPU before running. T4 is enough for the default small model. For a longer run, increase `TRAIN_STEPS`, `NUM_TRAIN_STATES`, and the task list.

In [1]:
from pathlib import Path
import os

# Local-first configuration. Override CASCADEGUARD_REPO if you launch the notebook
# from outside the repo root.
LOCAL_REPO_DIR = os.environ.get("CASCADEGUARD_REPO", "")
USE_LOCAL_CHECKOUT = True

# Training does not require the server. Set this True only if you want an API smoke test.
START_LOCAL_SERVER = False
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000

# Remote/API baselines are optional and disabled by default so the whole notebook
# can run before the new environment is pushed to Hugging Face.
RUN_REMOTE_SPACE_SMOKE = False
RUN_LLM_BASELINE = True
HF_SPACE_URL = "https://samarthdave0305-cascade-failure-env.hf.space"
GOOGLE_MODEL = "gemini-1.5-flash"
GROQ_MODEL = "llama-3.1-8b-instant"

# Full local GRPO pass. For a quick smoke test, temporarily set TRAIN_STEPS=10
# and NUM_TRAIN_STATES=32.
TRAIN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
USE_UNSLOTH = True
TRAIN_STEPS = 100
NUM_TRAIN_STATES = 128
NUM_GENERATIONS = 8
LORA_R = 32
LORA_ALPHA = 64
TRAIN_TASKS = [
    "task_easy",
    "task_medium",
    "task_gen_blackout",
    "task_hard",
    "task_cyberattack",
]
EVAL_TASKS = list(TRAIN_TASKS)
EVAL_SEEDS_PER_TASK = 1

OUTPUT_DIR = "cascadeguard_grpo_local"
RESULTS_CSV = "cascadeguard_eval_results.csv"

print("Configured CascadeGuard local GRPO training")
print("Model:", TRAIN_MODEL)
print("Train tasks:", TRAIN_TASKS)
print("Steps:", TRAIN_STEPS, "states:", NUM_TRAIN_STATES, "generations:", NUM_GENERATIONS)


Configured CascadeGuard local GRPO training
Model: Qwen/Qwen2.5-1.5B-Instruct
Train tasks: ['task_easy', 'task_medium', 'task_gen_blackout', 'task_hard', 'task_cyberattack']
Steps: 100 states: 128 generations: 8


## 2. Install Local Dependencies

This bootstrap installs the local package and verifies the ML stack in a fresh subprocess before training. It intentionally avoids loose `pip install -U transformers peft torch` upgrades, because that is what caused the `torch.fx.experimental.proxy_tensor` / TRL import mismatch.

Default behavior is `CASCADEGUARD_ML_INSTALL=auto`: reinstall the pinned GRPO stack only when imports fail or when `USE_UNSLOTH=True` and Unsloth is missing. Set `CASCADEGUARD_ML_INSTALL=force` for a clean Colab repair, or `skip` if you know the runtime is already healthy.

If this cell repairs Torch/TRL/Unsloth, restart the runtime/kernel, then rerun the notebook from the top.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path


def run(args, cwd=None):
    printable = " ".join(str(a) for a in args)
    print("$", printable)
    subprocess.run([str(a) for a in args], cwd=cwd, check=True)


def run_capture(args):
    return subprocess.run(
        [str(a) for a in args],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )


def find_project_root():
    if LOCAL_REPO_DIR:
        candidate = Path(LOCAL_REPO_DIR).expanduser().resolve()
        if (candidate / "cascade_guard" / "server" / "cascade_environment.py").exists():
            return candidate
        if (candidate / "server" / "cascade_environment.py").exists():
            return candidate.parent
        raise RuntimeError(f"CASCADEGUARD_REPO does not look like this repo: {candidate}")

    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "cascade_guard" / "server" / "cascade_environment.py").exists():
            return candidate
    raise RuntimeError(
        "Could not find the local CascadeGuard checkout. Run this notebook from the repo "
        "or set CASCADEGUARD_REPO to the folder that contains cascade_guard/."
    )


def ml_stack_healthy(require_unsloth=USE_UNSLOTH):
    probe = '''
import importlib.metadata as md
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig
if REQUIRE_UNSLOTH:
    from unsloth import FastLanguageModel
names = ["torch", "transformers", "trl", "peft", "accelerate"]
if REQUIRE_UNSLOTH:
    names.append("unsloth")
for name in names:
    try:
        print(f"{name}=={md.version(name)}")
    except Exception:
        print(f"{name} version unknown")
print("cuda_available=", torch.cuda.is_available())
'''.replace("REQUIRE_UNSLOTH", "True" if require_unsloth else "False")
    result = run_capture([sys.executable, "-c", probe])
    if result.returncode == 0:
        print("ML stack import check passed:")
        print(result.stdout.strip())
        return True
    print("ML stack import check failed; repair will run. Details:")
    print(result.stdout[-4000:])
    return False


PROJECT_ROOT = find_project_root()
PACKAGE_DIR = PROJECT_ROOT / "cascade_guard"
os.chdir(PROJECT_ROOT)

# Stable training stack for this notebook. Torch is installed first, then HF/TRL,
# then Unsloth. This avoids the broken state where new Transformers sees old Torch.
TORCH_INDEX_URL = os.environ.get(
    "CASCADEGUARD_TORCH_INDEX_URL",
    "https://download.pytorch.org/whl/cu126",
)
TORCH_PACKAGES = [
    "torch==2.7.0",
    "torchvision==0.22.0",
    "torchaudio==2.7.0",
]
ML_PACKAGES = [
    "accelerate==1.10.1",
    "bitsandbytes==0.48.1",
    "datasets==4.2.0",
    "peft==0.17.1",
    "sentencepiece==0.2.0",
    "transformers==4.57.1",
    "trl==0.24.0",
    "protobuf<6",
]
APP_PACKAGES = [
    "openenv-core[core]>=0.2.2",
    "pandas>=2.2",
    "matplotlib>=3.8",
    "groq>=0.9",
    "google-genai>=1.0",
    "uvicorn>=0.30",
    "fastapi>=0.115",
]

install_mode = os.environ.get("CASCADEGUARD_ML_INSTALL", "auto").strip().lower()
if install_mode not in {"auto", "force", "skip"}:
    raise ValueError("CASCADEGUARD_ML_INSTALL must be auto, force, or skip")

run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel", "packaging"])

needs_ml_repair = install_mode == "force" or (install_mode == "auto" and not ml_stack_healthy())
if install_mode == "skip":
    needs_ml_repair = False

if needs_ml_repair:
    print("Installing pinned Torch/HF/TRL stack. This can take several minutes on a fresh Colab runtime.")
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *TORCH_PACKAGES, "--index-url", TORCH_INDEX_URL])
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *ML_PACKAGES])
    if USE_UNSLOTH:
        run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "unsloth"])
        # Keep Unsloth's resolver from drifting the core GRPO stack after install.
        run([
            sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-deps",
            "accelerate==1.10.1", "peft==0.17.1", "transformers==4.57.1", "trl==0.24.0",
        ])
    if not ml_stack_healthy():
        raise RuntimeError("Pinned ML stack still does not import cleanly after reinstall; inspect the log above.")
    print("ML stack installed. Restart the runtime/kernel now, then rerun from the top.")
    if importlib.util.find_spec("google.colab") is not None:
        os.kill(os.getpid(), 9)
    raise SystemExit("Restart the Python kernel/runtime, then rerun the notebook from cell 1.")

run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *APP_PACKAGES])
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PACKAGE_DIR)])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Package dir:", PACKAGE_DIR)
print("Working directory:", os.getcwd())
print("Install mode:", install_mode)


$ c:\Users\Samarth Dave\Miniconda3\envs\rl_env\python.exe -m pip install -q --upgrade pip setuptools wheel packaging
ML stack import check failed; repair will run. Details:
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).
W0422 10:09:45.631000 59668 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Traceback (most recent call last):
  File "c:\Users\Samarth Dave\Miniconda3\envs\rl_env\lib\site-packages\trl\import_utils.py", line 156, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
  File "c:\Users\Samarth Dave\Miniconda3\envs\rl_env\lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
  File "<frozen importlib._bootstrap>", line 1050, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1027, in _find_and_load
  File "<

CalledProcessError: Command '['c:\\Users\\Samarth Dave\\Miniconda3\\envs\\rl_env\\python.exe', '-m', 'pip', 'install', '-q', '--upgrade', 'torch==2.7.0', 'torchvision==0.22.0', 'torchaudio==2.7.0', '--index-url', 'https://download.pytorch.org/whl/cu126']' returned non-zero exit status 1.

## 3. Optional Secrets

No secret is required for local environment training. Set `HF_TOKEN` only if the base model you choose requires authentication. Set `GROQ_API_KEY` or `GEMINI_API_KEY` only if you enable `RUN_LLM_BASELINE`.


In [ ]:
# Optional: set tokens here or through your shell/Colab secrets.
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["GROQ_API_KEY"] = "gsk_..."
# os.environ["GEMINI_API_KEY"] = "..."
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("GROQ_API_KEY set:", bool(os.environ.get("GROQ_API_KEY")))
print("GEMINI_API_KEY set:", bool(os.environ.get("GEMINI_API_KEY")))


In [ ]:
import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def set_secret(name, fallback_prompt=False):
    value = os.environ.get(name)
    if not value and userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value and fallback_prompt:
        value = getpass.getpass(f"Enter {name}: ")
    if value:
        os.environ[name] = value
    return value


# Pull optional secrets from Colab only when available. No interactive prompts by default.
hf_token = set_secret("HF_TOKEN", fallback_prompt=False)
groq_key = set_secret("GROQ_API_KEY", fallback_prompt=False)
gemini_key = set_secret("GEMINI_API_KEY", fallback_prompt=False)
if gemini_key:
    os.environ["GOOGLE_API_KEY"] = gemini_key

print("Local training ready. Optional secrets loaded:", {
    "HF_TOKEN": bool(hf_token),
    "GROQ_API_KEY": bool(groq_key),
    "GEMINI_API_KEY": bool(gemini_key),
})


In [ ]:
import sys

# Keep local checkout first so imports use your unpushed environment code.
sys.path = [p for p in sys.path if "site-packages" not in p or p != str(PROJECT_ROOT)]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Using project root:", PROJECT_ROOT)


In [ ]:
import cascade_guard
print("Import successful:", cascade_guard.__file__)


In [ ]:
# No-op compatibility cell kept intentionally so older notebook references still run.
print("Local package import is configured.")


## 4. Local OpenEnv Sanity Check

This checks the package, tasks, step/reset loop, action metadata, and grader score.

In [ ]:
import json, time, math, re, statistics, random
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

from cascade_guard.models import CascadeAction
from cascade_guard.server.cascade_environment import CascadeEnvironment
from cascade_guard.tasks import TASK_CONFIGS, TASK_SEED_SPLITS
from cascade_guard.training.cot_prompt import build_system_prompt, build_user_prompt, parse_action_from_response

for task_id in ["task_easy", "task_medium", "task_hard", "task_gen_blackout", "task_cyberattack"]:
    env = CascadeEnvironment()
    obs = env.reset(
        task_id=task_id,
        seed=TASK_SEED_SPLITS[task_id]["train"][0],
        reward_mode="grpo",
        training_mode=True,
    )
    legal = env.get_legal_actions()
    obs = env.step(CascadeAction(action_type="wait", target_node_id=None))
    metadata = obs.metadata or {}
    print(
        task_id,
        "nodes=", len(obs.nodes),
        "stress=", obs.upcoming_stress_level,
        "legal=", len(legal),
        "reward=", round(float(obs.reward), 4),
        "raw=", metadata.get("raw_reward"),
        "mode=", metadata.get("reward_mode"),
    )

print("Local GRPO environment sanity check complete")


## 5. Optional Local Server Smoke Test

Training uses `CascadeEnvironment` in-process, so you do not need the server. If you want to test the OpenEnv HTTP API locally, set `START_LOCAL_SERVER = True` in the settings cell or run this command in a terminal from the repo root:

```bash
python -m uvicorn cascade_guard.server.app:app --host 127.0.0.1 --port 8000
```

Then open `http://127.0.0.1:8000/health`.


In [ ]:
import subprocess
import time
import urllib.request

server_proc = None
if START_LOCAL_SERVER:
    cmd = [
        sys.executable,
        "-m",
        "uvicorn",
        "cascade_guard.server.app:app",
        "--host",
        SERVER_HOST,
        "--port",
        str(SERVER_PORT),
    ]
    print("Starting local server:", " ".join(cmd))
    server_proc = subprocess.Popen(cmd, cwd=str(PROJECT_ROOT))
    health_url = f"http://{SERVER_HOST}:{SERVER_PORT}/health"
    for _ in range(30):
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                print("Local server health:", response.read().decode("utf-8"))
                break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError(f"Server did not become healthy at {health_url}")
else:
    print("Server not started. Training will use the local environment in-process.")
    print(f"Manual command: python -m uvicorn cascade_guard.server.app:app --host {SERVER_HOST} --port {SERVER_PORT}")


## 6. Baseline Policies

The heuristic baseline is deterministic. The Gemini baseline uses `gemini-2.5-flash` and returns a strict JSON action.

In [ ]:
try:
    from groq import Groq
except Exception:
    Groq = None
try:
    from google.genai import types
except Exception:
    types = None

groq_client = None
if RUN_LLM_BASELINE and Groq is not None and os.environ.get("GROQ_API_KEY"):
    groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def compact_obs_prompt(obs):
    diagnostics = obs.diagnostics
    notable = []
    for n in obs.nodes:
        if (not n.is_operational) or n.health < 0.75 or n.load > 0.75 or n.observation_delayed or n.is_critical:
            notable.append({
                "id": n.node_id,
                "sector": n.sector,
                "health": round(float(n.health), 2),
                "load": round(float(n.load), 2),
                "operational": bool(n.is_operational),
                "hardened": bool(n.is_hardened),
                "delayed": bool(n.observation_delayed),
                "critical": bool(n.is_critical),
            })
    payload = {
        "task_id": obs.task_id,
        "step": obs.step,
        "max_steps": obs.max_steps,
        "budget_remaining": round(float(obs.budget_remaining), 2),
        "weather": obs.weather_forecast,
        "active_failures": obs.active_failures,
        "pending_recoveries": obs.pending_recoveries,
        "sector_summary": {k: round(float(v), 2) for k, v in obs.sector_summary.items()},
        "diagnostics": diagnostics.model_dump() if diagnostics else {},
        "notable_nodes": notable,
        "edges": [e.model_dump() for e in obs.edges],
    }
    return json.dumps(payload, separators=(",", ":"))

def action_from_json_text(text, obs):
    try:
        match = re.search(r"\{.*\}", text, re.S)
        data = json.loads(match.group(0) if match else text)
        return CascadeAction(action_type=data.get("action_type", "wait"), target_node_id=data.get("target_node_id"))
    except Exception:
        return parse_action_from_response(text, obs)

def heuristic_policy(obs):
    diag = obs.diagnostics
    if diag and diag.recommended_recovery_order:
        return CascadeAction(action_type="recover", target_node_id=diag.recommended_recovery_order[0])
    failed = [n for n in obs.nodes if not n.is_operational and n.node_id not in set(obs.pending_recoveries)]
    if failed:
        failed.sort(key=lambda n: (not n.is_critical, n.health))
        return CascadeAction(action_type="recover", target_node_id=failed[0].node_id)
    delayed = [n for n in obs.nodes if n.observation_delayed]
    if delayed and obs.budget_remaining >= 1.0:
        delayed.sort(key=lambda n: (not n.is_critical, n.health))
        return CascadeAction(action_type="coordinate", target_node_id=delayed[0].node_id)
    at_risk = []
    if diag and diag.at_risk_nodes:
        node_map = {n.node_id: n for n in obs.nodes}
        at_risk = [node_map[nid] for nid in diag.at_risk_nodes if nid in node_map]
    if obs.budget_remaining >= 2.0 and (obs.weather_forecast != "clear" or obs.step < 5):
        candidates = [n for n in (at_risk or obs.nodes) if n.is_operational and not n.is_hardened]
        if candidates:
            candidates.sort(key=lambda n: (not n.is_critical, n.health))
            return CascadeAction(action_type="harden", target_node_id=candidates[0].node_id)
    overloaded = [n for n in obs.nodes if n.is_operational and (not n.is_critical) and n.sector != "hospital" and n.load > 0.85]
    if overloaded:
        overloaded.sort(key=lambda n: n.load, reverse=True)
        return CascadeAction(action_type="shed_load", target_node_id=overloaded[0].node_id)
    return CascadeAction(action_type="wait", target_node_id=None)

def groq_policy(obs):
    prompt = (
        "You are a critical-infrastructure resilience agent. Return ONLY JSON with keys "
        "action_type and target_node_id. action_type must be one of harden, recover, "
        "shed_load, coordinate, isolate, wait. Never shed_load hospitals or critical nodes. "
        "Recover upstream dependencies before downstream nodes.\n\n"
        "Observation JSON:\n"
        + compact_obs_prompt(obs)
    )

    if groq_client is None:
        return heuristic_policy(obs), 0

    try:
        completion = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",  # fast + cheap
            messages=[
                {"role": "system", "content": "Return only JSON."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
        )

        text = completion.choices[0].message.content
        return action_from_json_text(text, obs), 0

    except Exception as e:
        print("Groq error:", e)
        return heuristic_policy(obs), 0


## 7. Evaluation Helpers

The evaluation table tracks more than reward: score, success, invalid actions, budget use, critical failures, and response speed.

In [ ]:
def run_local_episode(policy_name, policy_fn, task_id, seed):
    env = CascadeEnvironment()
    obs = env.reset(task_id=task_id, seed=seed, scenario_split="validation")
    invalid_actions = 0
    critical_failures = 0
    rewards = []
    latencies = []
    actions = []
    done = False
    while not done:
        t0 = time.perf_counter()
        out = policy_fn(obs)
        if isinstance(out, tuple):
            action, elapsed_ms = out
        else:
            action, elapsed_ms = out, (time.perf_counter() - t0) * 1000.0
        latencies.append(float(elapsed_ms))
        actions.append({"action_type": action.action_type, "target_node_id": action.target_node_id})
        obs = env.step(action)
        rewards.append(float(obs.reward))
        metadata = obs.metadata or {}
        if not metadata.get("action_valid", True):
            invalid_actions += 1
        if obs.diagnostics:
            critical_failures += len(obs.diagnostics.critical_failures)
        done = bool(obs.done)
    state = env.state
    avg_sector = sum(state.sector_health.values()) / max(len(state.sector_health), 1)
    return {
        "policy": policy_name,
        "task_id": task_id,
        "seed": seed,
        "score": float(state.score),
        "success": bool(avg_sector >= 0.5 and invalid_actions == 0),
        "steps": int(state.step_count),
        "mean_reward": statistics.mean(rewards) if rewards else 0.0,
        "invalid_actions": invalid_actions,
        "invalid_action_rate": invalid_actions / max(len(rewards), 1),
        "budget_spent": float(TASK_CONFIGS[task_id]["budget"] - state.budget_remaining),
        "critical_failures": critical_failures,
        "avg_response_ms": statistics.mean(latencies) if latencies else 0.0,
        "actions": json.dumps(actions),
    }

def evaluate_policy(policy_name, policy_fn, tasks=EVAL_TASKS, seeds_per_task=EVAL_SEEDS_PER_TASK):
    rows = []
    for task_id in tasks:
        seeds = TASK_SEED_SPLITS[task_id]["validation"][:seeds_per_task]
        for seed in seeds:
            rows.append(run_local_episode(policy_name, policy_fn, task_id, seed))
    return pd.DataFrame(rows)

heuristic_df = evaluate_policy("heuristic", heuristic_policy)
display(heuristic_df.drop(columns=["actions"]))

In [ ]:
if RUN_LLM_BASELINE:
    try:
        time.sleep(2)
        gemini_df = evaluate_policy(
            "groq",
            groq_policy,
            tasks=EVAL_TASKS[:1],
            seeds_per_task=1,
        )
    except Exception as exc:
        print("LLM baseline skipped after error:", exc)
        gemini_df = pd.DataFrame()
else:
    print("Skipping external LLM baseline. Set RUN_LLM_BASELINE=True to enable it.")
    gemini_df = pd.DataFrame()


## 8. Build GRPO Training States

Each GRPO sample is a real CascadeGuard observation. The reward function reconstructs the environment from task, seed, and previous actions, applies the generated action, and returns the environment reward plus small format/validity shaping.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = build_system_prompt(grpo_mode=True) + "\nReturn exactly one <action>...</action> tag."

def make_training_prompt(obs, env=None):
    base = SYSTEM_PROMPT + "\n\n" + build_user_prompt(obs)
    if env is not None and hasattr(env, "get_legal_actions"):
        legal_strs = []
        for action in env.get_legal_actions()[:12]:
            action_type = action.get("action_type", "wait")
            target = action.get("target_node_id")
            legal_strs.append(f"{action_type}({target if target else 'null'})")
        base += "\n\nLegal actions now: " + " | ".join(legal_strs)
    return base

def collect_training_states(num_states=NUM_TRAIN_STATES, tasks=TRAIN_TASKS):
    rows = []
    for task_id in tasks:
        for seed in TASK_SEED_SPLITS[task_id]["train"]:
            env = CascadeEnvironment()
            obs = env.reset(
                task_id=task_id,
                seed=seed,
                scenario_split="train",
                reward_mode="grpo",
                training_mode=True,
            )
            history = []
            done = False
            while not done and len(rows) < num_states:
                rows.append({
                    "prompt": make_training_prompt(obs, env),
                    "task_id": task_id,
                    "seed": int(seed),
                    "history_json": json.dumps(history),
                })
                action = heuristic_policy(obs)
                history.append({"action_type": action.action_type, "target_node_id": action.target_node_id})
                obs = env.step(action)
                done = bool(obs.done)
            if len(rows) >= num_states:
                return rows
    return rows

train_rows = collect_training_states()
train_ds = Dataset.from_list(train_rows)
print(train_ds)
print(train_ds[0]["prompt"][:900])



In [ ]:
def completion_to_text(completion):
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion:
        last = completion[-1]
        if isinstance(last, dict):
            return str(last.get("content", ""))
    return str(completion)

def ensure_list(value, n, default):
    if value is None:
        return [default] * n
    if isinstance(value, (str, int, float)):
        return [value] * n
    return list(value)

def cascade_grpo_reward(prompts, completions, task_id=None, seed=None, history_json=None, **kwargs):
    rewards = []
    n = len(completions)
    task_ids = ensure_list(task_id, n, "task_easy")
    seeds = ensure_list(seed, n, 42)
    histories = ensure_list(history_json, n, "[]")
    for completion, tid, sd, hist_text in zip(completions, task_ids, seeds, histories):
        env = CascadeEnvironment()
        obs = env.reset(
            task_id=str(tid),
            seed=int(sd),
            scenario_split="train",
            reward_mode="grpo",
            training_mode=True,
        )
        try:
            history = json.loads(hist_text or "[]")
        except Exception:
            history = []
        for item in history:
            try:
                obs = env.step(CascadeAction(**item))
            except Exception:
                break

        text = completion_to_text(completion)
        action = parse_action_from_response(text, obs)
        legal = env.get_legal_actions() if hasattr(env, "get_legal_actions") else []
        legal_types = {a["action_type"] for a in legal}

        next_obs = env.step(action)
        metadata = next_obs.metadata or {}
        reward = float(metadata.get("raw_reward", 0.0))

        if "<action>" in text.lower():
            reward += 0.08
        else:
            reward -= 0.15

        if not metadata.get("action_valid", True):
            reward -= 0.40
            if legal_types - {"wait"}:
                reward -= 0.15

        if action.action_type == "wait":
            better_actions = [a for a in legal if a["action_type"] != "wait"]
            if len(better_actions) >= 4:
                reward -= 0.50
            elif len(better_actions) >= 2:
                reward -= 0.30
            elif len(better_actions) >= 1:
                reward -= 0.15

        if action.action_type == "recover" and metadata.get("action_valid", True):
            reward += 0.25

        sector_summary = env._compute_sector_summary()
        avg_health = sum(sector_summary.values()) / max(len(sector_summary), 1)
        reward += 0.10 * (avg_health - 0.5)

        rewards.append(float(max(-3.0, min(3.0, reward))))

    import statistics
    if len(rewards) >= 2:
        mu = statistics.mean(rewards)
        sd = statistics.pstdev(rewards)
        if sd > 0.01:
            rewards = [(r - mu) / sd for r in rewards]
        else:
            rewards = [0.0] * len(rewards)
    return rewards

# Quick reward smoke test.
print(cascade_grpo_reward(
    [train_ds[0]["prompt"]],
    ["<action>harden(POWER_GEN_1)</action>"],
    task_id=[train_ds[0]["task_id"]],
    seed=[train_ds[0]["seed"]],
    history_json=[train_ds[0]["history_json"]],
))


## 9. GRPO Train

This uses TRL `GRPOTrainer`. The install cell above first verifies that `torch`, `transformers`, `trl`, and `peft` import together. With `USE_UNSLOTH=True`, it also installs and tries Unsloth 4-bit loading; if Unsloth itself cannot load in the current runtime, this cell falls back to standard Transformers + PEFT so training still runs.


In [ ]:
import importlib.metadata as md
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
for package in ["torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "unsloth"]:
    try:
        print(f"{package}=={md.version(package)}")
    except Exception:
        print(f"{package}: not installed")


In [ ]:
# Compatibility no-op: older notebook copies had a duplicate CUDA check here.


In [ ]:
# Compatibility no-op: GRPO imports happen in the training cell after the stack check.


In [ ]:
import os

import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer


def make_lora_config():
    return LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )


def load_standard_model():
    tokenizer = AutoTokenizer.from_pretrained(TRAIN_MODEL, token=os.environ.get("HF_TOKEN") or None)
    model_kwargs = {
        "device_map": "auto" if torch.cuda.is_available() else None,
        "token": os.environ.get("HF_TOKEN") or None,
    }
    if torch.cuda.is_available():
        model_kwargs["torch_dtype"] = torch_dtype
        try:
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch_dtype,
            )
            print("Using standard Transformers 4-bit BitsAndBytes load.")
        except Exception as exc:
            print("BitsAndBytes 4-bit config unavailable; loading standard precision:", repr(exc))
    else:
        model_kwargs["torch_dtype"] = torch.float32
    model_kwargs = {key: value for key, value in model_kwargs.items() if value is not None}
    model = AutoModelForCausalLM.from_pretrained(TRAIN_MODEL, **model_kwargs)
    return model, tokenizer, make_lora_config(), "transformers_peft"


torch_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

model = tokenizer = peft_config = None
backend = None

if USE_UNSLOTH:
    try:
        from unsloth import FastLanguageModel

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=TRAIN_MODEL,
            max_seq_length=2048,
            load_in_4bit=True,
            dtype=None,
            token=os.environ.get("HF_TOKEN") or None,
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_R,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=LORA_ALPHA,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=42,
        )
        backend = "unsloth_4bit"
        print("Loaded model with Unsloth 4-bit QLoRA.")
    except Exception as exc:
        print("Unsloth load failed; falling back to Transformers + PEFT:", repr(exc))
        model, tokenizer, peft_config, backend = load_standard_model()
else:
    model, tokenizer, peft_config, backend = load_standard_model()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

precision_kwargs = {
    "bf16": torch_dtype == torch.bfloat16,
    "fp16": torch_dtype == torch.float16,
}

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    max_steps=TRAIN_STEPS,
    learning_rate=5e-6,
    per_device_train_batch_size=NUM_GENERATIONS,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_completion_length=128,
    temperature=0.7,
    logging_steps=1,
    save_steps=max(TRAIN_STEPS, 1),
    report_to="none",
    remove_unused_columns=False,
    **precision_kwargs,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=cascade_grpo_reward,
    train_dataset=train_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("Training backend:", backend)
trainer.train()
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print("Saved local GRPO checkpoint to", f"{OUTPUT_DIR}/final")


## 10. Evaluate GRPO Policy

This runs full local episodes with the trained policy and compares against the earlier baselines.

In [ ]:
def grpo_policy(obs):
    prompt = make_training_prompt(obs)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return parse_action_from_response(text, obs), elapsed_ms

grpo_df = evaluate_policy("grpo_local", grpo_policy)
frames = [df for df in [heuristic_df, gemini_df, grpo_df] if df is not None and not df.empty]
all_results = pd.concat(frames, ignore_index=True)
summary = all_results.groupby("policy").agg(
    runs=("score", "count"),
    mean_score=("score", "mean"),
    success_rate=("success", "mean"),
    invalid_action_rate=("invalid_action_rate", "mean"),
    mean_budget_spent=("budget_spent", "mean"),
    mean_critical_failures=("critical_failures", "mean"),
    avg_response_ms=("avg_response_ms", "mean"),
).reset_index()

display(all_results.drop(columns=["actions"]))
display(summary)


## 11. Save Results

The CSV is the artifact to show before the hackathon. It contains per-episode metrics and the action trace JSON.

In [ ]:
all_results.to_csv(RESULTS_CSV, index=False)
summary.to_csv("cascadeguard_eval_summary.csv", index=False)
print("Saved", RESULTS_CSV)
print("Saved cascadeguard_eval_summary.csv")

try:
    import matplotlib.pyplot as plt
    ax = summary.plot(kind="bar", x="policy", y="mean_score", legend=False, figsize=(7, 4), ylim=(0, 1))
    ax.set_title("CascadeGuard Policy Score")
    ax.set_ylabel("Mean score")
    plt.tight_layout()
    plt.savefig("cascadeguard_policy_score.png", dpi=160)
    plt.show()
    print("Saved cascadeguard_policy_score.png")
except Exception as exc:
    print("Plot skipped:", repr(exc))

## 12. After The Local Run Passes

Recommended next checks before pushing:

- Keep `TRAIN_STEPS = 200` and increase `NUM_TRAIN_STATES` to 500+ for a more serious run.
- Increase `EVAL_SEEDS_PER_TASK` to 2 or 3 for a less noisy estimate.
- Compare `grpo_local` against the heuristic and optional Groq baseline.
- Inspect `cascadeguard_eval_results.csv`, `cascadeguard_eval_summary.csv`, and `cascadeguard_policy_score.png`.
- Push the environment only after the local notebook uses `reward_mode="grpo"`, `get_legal_actions()`, `isolate`, and `training_mode=True` without errors.

Server reminder if you need API testing:

```bash
python -m uvicorn cascade_guard.server.app:app --host 127.0.0.1 --port 8000
```
